In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from tsfresh import extract_features
from sklearn.metrics import root_mean_squared_error 

# I. DATA EXPLORATION

In [4]:
x_train = pd.read_csv("x_train.csv",header=None)
y_train = pd.read_csv("y_train.csv",header=None)

In [5]:
df_plot = pd.concat([x_train.iloc[:500,:], y_train.iloc[:500,:]], axis=0).reset_index(drop=True)
df = df_plot.iloc[:100,:]

# Set up the plot
fig, ax = plt.subplots(figsize=(10, 5))
line, = ax.plot([], [], lw=2)
highlight, = ax.plot([], [], color='red', lw=2)  # To highlight the last 5 seconds
# highlight_green, = ax.plot([], [], color='green') 
ax.set_xlim(0, df.shape[1] - 1)  # Set x-limits based on the number of columns
ax.set_ylim(df.min().min(), df.max().max())  # Set y-limits based on the data range
ax.set_title('65-Second Time Series Animation')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Value')
ax.grid(True)

# The animate function
def animate(i):
    x = range(df.shape[1])  # Assuming the columns represent time in seconds
    y = df.iloc[i]  # Get the data for the i-th 65-second window
    
    # Update the main line
    line.set_data(x, y)

    # Highlight the last 10 to 5 seconds in green
    # highlight_green.set_data(x[-10:-5], y[-10:-5])
    
    # Highlight the last 5 seconds (last 5 columns) differently
    highlight.set_data(x[-5:], y[-5:])  # Plot the last 5 points in red
    
    ax.set_title(f'65-Second Window {i + 1}')
    return line, highlight

# Create the animation
ani = FuncAnimation(fig, animate, frames=len(df), blit=True, interval=1000)

# Close the plot to avoid displaying the static figure
plt.close(fig)

# Display the animation in the notebook
HTML(ani.to_jshtml())


# II. TRANFORMATION DES DONNEES

In [6]:
x_train = pd.read_csv("x_train.csv",header=None)
y_train = pd.read_csv("y_train.csv",header=None)

In [7]:
# number of lines to keep
m = 500
x_train = x_train.iloc[:m,:]

## II.1 Création des features

In [8]:
# Step 1: Reshape x_train to long format for tsfresh
x_train_long = x_train.melt(ignore_index=False).reset_index()
x_train_long.columns = ['id', 'time', 'value']  # Rename columns to match tsfresh expectations

# Step 2: Extract features
extracted_features = extract_features(
    x_train_long, 
    column_id='id', 
    column_sort='time', 
    column_value='value',
    default_fc_parameters=None  # You can specify a set of features or leave it as None for the full set
)

Feature Extraction: 100%|██████████| 20/20 [00:05<00:00,  3.80it/s]


In [9]:
extracted_features = extracted_features.dropna(axis = 1)

In [10]:
# Calculate the 75th quantile of all values in the DataFrame
threshold = y_train.quantile(0.75).max()

y_train = y_train.iloc[:m,:]

# Create the is_pic column: 1 if any value in the row is above the threshold, else 0
y_train['is_pic'] = y_train.apply(lambda row: 1 if any(row > threshold) else 0, axis=1)

# del y_train['is_pic']

In [11]:
y_select = y_train.iloc[:m,-1]
# y_select = y_train.iloc[:50,:].mean(axis=1)

In [12]:
# from tsfresh import select_features

# # Perform feature selection
# selected_features = select_features(extracted_features, y_select)
# selected_features.head()

## II.2 Selection des features

In [13]:
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier  # or RandomForestRegressor for regression
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split


# # Step 1: Clean data by replacing infinities and NaNs
# extracted_features.replace([np.inf, -np.inf], np.nan, inplace=True)
# extracted_features.fillna(extracted_features.mean(), inplace=True)

# X_train, X_test, y_train_split, y_test_split = train_test_split(extracted_features, y_select, test_size=0.2, random_state=42)

# # Step 3: Define the model
# model = RandomForestClassifier(random_state=42)  # Use RandomForestRegressor for regression tasks

# # Step 4: Apply RFE
# # Specify the number of features to select; you can adjust based on the number of features you want to keep
# n_features_to_select = 10  # You can adjust this based on your preference
# rfe = RFE(estimator=model, n_features_to_select=n_features_to_select)
# X_train_rfe = rfe.fit_transform(X_train, y_train_split)

# # Get the selected feature names
# selected_features = X_train.columns[rfe.support_]
# selected_features_df = extracted_features[selected_features]

# selected_features

In [14]:
# # Step 1: Split the data into training and testing sets
# X_train, X_test, y_train_split, y_test_split = train_test_split(selected_features_df, y_select, test_size=0.2, random_state=42)

# # Step 2: Train a Random Forest on the selected features
# model = RandomForestClassifier(random_state=42)
# model.fit(X_train, y_train_split)

# # Step 3: Make predictions on the test set
# y_pred = model.predict(X_test)

# # Step 4: Calculate the accuracy
# accuracy = accuracy_score(y_test_split, y_pred)
# print(f"Accuracy of the Random Forest model with selected features: {accuracy:.2f}")

In [16]:
# # Step 1: Standardize the features
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(extracted_features)

# # Step 2: Apply PCA
# pca = PCA()
# X_pca = pca.fit_transform(X_scaled)

# # Step 3: Plot the explained variance ratio
# explained_variance_ratio = pca.explained_variance_ratio_
# cumulative_variance_ratio = explained_variance_ratio.cumsum()

# # Create a plot of the explained variance
# plt.figure(figsize=(10, 6))

# # Bar plot for explained variance of each principal component
# plt.bar(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, alpha=0.6, label='Individual Explained Variance')
# # Line plot for cumulative explained variance
# plt.step(range(1, len(cumulative_variance_ratio) + 1), cumulative_variance_ratio, where='mid', label='Cumulative Explained Variance', color='red')

# plt.xlabel('Principal Component')
# plt.ylabel('Explained Variance Ratio')
# plt.title('PCA - Explained Variance by Principal Components')
# plt.legend()
# plt.show()

In [17]:
my_list = extracted_features.columns  # Example list

with open("output.txt", "w") as file:
    for item in my_list:
        file.write(f"{item}\n")  # Write each item on a new line

In [18]:
# Features that make the most sense :
selected_features = [
    "value__mean",
    "value__median",
    "value__standard_deviation",
    "value__variance",
    "value__skewness",
    "value__kurtosis",
    "value__root_mean_square",
    "value__abs_energy",
    "value__mean_abs_change",
    "value__count_above_mean",
    "value__count_below_mean",
    "value__longest_strike_above_mean",
    "value__longest_strike_below_mean",
    "value__autocorrelation__lag_1",
    "value__partial_autocorrelation__lag_1",
    "value__approximate_entropy__m_2__r_0.1",
    "value__linear_trend__attr_\"slope\"",
    "value__fft_coefficient__attr_\"real\"__coeff_1",
    "value__time_reversal_asymmetry_statistic__lag_1"
]


In [19]:
# Now I want to create a function that will take the explainative variable and will return it with the new features
from sklearn.preprocessing import MinMaxScaler

def process_data(data,choice=0):

    # Initialize the MinMaxScaler with your custom feature range
    scaler = MinMaxScaler()

    # Fit and transform x_train, keeping it as a DataFrame
    data_scaled = scaler.fit_transform(data)

    # Convert back to DataFrame, preserving column names
    data = pd.DataFrame(data_scaled, columns=x_train.columns)

    # Step 1: Reshape x_train to long format for tsfresh
    x_train_long = data.melt(ignore_index=False).reset_index()
    x_train_long.columns = ['id', 'time', 'value']  # Rename columns to match tsfresh expectations

    # Step 2: Extract features
    extracted_features = extract_features(
        x_train_long, 
        column_id='id', 
        column_sort='time', 
        column_value='value',
        default_fc_parameters=None)
    
    if choice == 0:
        x_train_long = extracted_features[selected_features]
        scaler2 = StandardScaler()
        x_train_long_scaled = scaler2.fit_transform(x_train_long)
        # Convert back to DataFrame, preserving column names
        x_train_long = pd.DataFrame(x_train_long_scaled, columns=x_train_long.columns)
        new_features = x_train_long

    if choice == 1:
        pca = PCA()
        X_pca = pca.fit_transform(data)
        pca_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
        scaler2 = StandardScaler()
        pca_scaled = scaler2.fit_transform(pca_df)
        pca_df = pd.DataFrame(pca_scaled, columns=pca_df.columns)
        pca_df = pca_df.iloc[:,:60]
        new_features = pca_df

    new_data = pd.concat([data,new_features], axis=1)
    
    return new_data

a  = process_data(x_train.iloc[:500,:],0)

Feature Extraction:   0%|          | 0/20 [00:00<?, ?it/s]

Feature Extraction: 100%|██████████| 20/20 [00:05<00:00,  3.59it/s]


# III. Modeling (IA)

## III.1 xgboost

In [20]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib

In [74]:
x_train = pd.read_csv("x_train.csv",header=None)
y_train = pd.read_csv("y_train.csv",header=None)

In [75]:
def scale_data(df):
    minim = 176
    maxim = 381085

    df_scaled = (df - minim)/(maxim - minim)

    return df_scaled

def de_scale_data(df):
    minim = 176
    maxim = 381085

    df_de_scaled = df * (maxim - minim) + minim

    return df_de_scaled

In [76]:
x_train = pd.read_csv("X_process_train_O.csv")
x_train = x_train.iloc[:,1:]

In [80]:
y_train_scaled = scale_data(y_train)
X = x_train.iloc[:30000,1:]
y = y_train_scaled.iloc[:30000,:].copy()

In [81]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Dictionary to store each model file path and predictions
model_paths = {}
predictions = []


# Train an XGBoost model for each target column, save the model, and predict
for col in y.columns:
    # Define and train the model
    model = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse')
    model.fit(X_train, y_train[col])
    
    # Save the model
    model_path = f"xgb_model_{col}.joblib"
    joblib.dump(model, model_path)
    model_paths[col] = model_path
    
    # Apply the prediction on the test set
    y_pred = model.predict(X_test)
    predictions.append(y_pred)

# Concatenate predictions for each of the columns into a DataFrame
y_pred_df = pd.DataFrame(predictions).T  # Transpose to align each prediction row-wise
y_pred_df.columns = y.columns  # Name the columns the same as y for easier reference
y_pred_df.index = X_test.index  # Align the index with X_test for comparison if needed

# Calculate and display the mean squared error for each column
for i, col in enumerate(y.columns):
    mse = mean_squared_error(y_test[col], y_pred_df[col])
    print(f"Mean Squared Error for {col}: {mse:.2f}")

ValueError: Found input variables with inconsistent numbers of samples: [30000, 16000]

In [38]:
a = de_scale_data(X_test.iloc[:100,:60])
b = de_scale_data(y_test.iloc[:100,:])
c = de_scale_data(y_pred_df.iloc[:100,:])

df_plot = pd.concat([a,b,c], axis=1).reset_index(drop=True)
df = df_plot

# Set up the plot
fig, ax = plt.subplots(figsize=(10, 5))
line, = ax.plot([], [], lw=2)
scatt = ax.scatter([], [], color='blue')
highlight, = ax.plot([], [], color='red', lw=2)
highlight_red = ax.scatter([], [], color='red', label="Last 5 Seconds")  # To highlight the last 5 seconds
highlight_green, = ax.plot([], [], color='green') 
high =  ax.scatter([], [], color='green')
ax.set_xlim(0, df.shape[1] - 1)  # Set x-limits based on the number of columns
ax.set_ylim(df.min().min(), df.max().max())  # Set y-limits based on the data range
ax.set_title('65-Second Time Series Animation')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Bit/s')
ax.grid(True)

# The animate function
def animate(i):
    x = range(df.shape[1]-5)  # Assuming the columns represent time in seconds
    y = df.iloc[i]  # Get the data for the i-th 65-second window
    
    # Update the main line
    line.set_data(x[:-4], y[:-9])

    scatt.set_offsets([(x_, y_) for x_, y_ in zip(range(0, 60), y[:60])])

    highlight_red.set_offsets([(x_, y_) for x_, y_ in zip(range(60, 65), y[-5:])])

    # Highlight the last 5 seconds (last 5 columns) differently
    highlight.set_data(x[-5:], y[-5:])  # Plot the last 5 points in red
    
    # Highlight the last 10 to 5 seconds in green
    highlight_green.set_data(x[-5:], y[-10:-5])
    high.set_offsets([(x_, y_) for x_, y_ in zip(range(60, 65), y[-10:-5])])
    
    ax.set_title(f'65-Second Window {i + 1}')
    return line, highlight

# Create the animation
ani = FuncAnimation(fig, animate, frames=len(df), blit=True, interval=1000)

# Close the plot to avoid displaying the static figure
plt.close(fig)

# Display the animation in the notebook
HTML(ani.to_jshtml())

In [39]:
import utils 
import importlib
from utils import eval_model
importlib.reload(utils)

def eval_model(pred_csv, y_csv):
    pred_test = pd.read_csv(pred_csv, sep=',', header=None, index_col=False).to_numpy()
    y_test = pd.read_csv(y_csv, sep=',', header=None, index_col=False).to_numpy() 
    min_d, max_d = 176, 381085
    min_val, max_val = int(min_d), int(max_d)
    global_nrmse = 0.0
    i = 0
    for pred, gt in zip(pred_test, y_test):
        if i == 0:
            i +=1
            continue
        global_nrmse += root_mean_squared_error(gt, pred) / (max_val - min_val)
    global_nrmse /= len(pred_test)
    print(f"global test nRMSE: {global_nrmse:.8f}")

def evaluate_model(y_test, y_pred_df):
    de_scale_data(y_test).to_csv("y_test1.csv")
    de_scale_data(y_pred_df).to_csv("y_pred1.csv")

    path1 = "y_test1.csv"
    path2 = "y_pred1.csv"

    eval_model(path1,path2)

In [82]:
evaluate_model(y_test, y_pred_df)

global test nRMSE: 0.11929073


## III.2 Lasso Benchmark 2

In [53]:
# Try to predict the mean
y_lasso = y.mean(axis= 1)
y_lasso = pd.DataFrame(y_lasso)
X_filled = X.fillna(X.mean())

In [68]:
y_lasso2 = scale_data(y_lasso)
X_filled2 = scale_data(X_filled).iloc[:,45:]

In [ ]:
# Split the data into training and testing sets
# X_new = X_filled.to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X_filled2, y_lasso2, test_size=0.2, random_state=42)

In [71]:
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

# Initialize Lasso with a high alpha value
alpha = 0.5  # Set a high regularization parameter
lasso = Lasso(alpha=alpha, random_state=42)

# Fit the model on the training data
lasso.fit(X_train, y_train)

# Predict on the test set
y_pred = lasso.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error on the test set: {mse:.2f}")

# Retrieve coefficients
lasso_coefficients = lasso.coef_
print(f"Lasso Coefficients: {lasso_coefficients}")


Mean Squared Error on the test set: 0.00
Lasso Coefficients: [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0. -0. -0.  0.  0.  0.  0. -0.  0.  0.  0.  0.  0.  0.  0.  0.]


In [72]:
y_train = pd.read_csv("y_train.csv",header=None)
y_test_lasso = y_train.loc[y_test.index]
y_test_lasso

,0,1,2,3,4
6252,6073,6981,71901,6033,69709
4684,3933,6993,60501,40973,19277
1731,23808,176,8608,3392,18912
4742,57984,664,139752,8988,42216
4521,176,24888,16208,38048,23664
...,...,...,...,...,...
6412,62168,3108,3104,2152,25424
8285,123304,77512,61744,91072,34776
7853,65190,54022,30222,76590,21854
1095,20056,66984,122688,74944,145168


In [57]:
a = de_scale_data(X_test.iloc[:200,:60])
b = y_test_lasso.iloc[:200,:]
y_pred_df = pd.DataFrame(y_pred)
y_pred_df.index = y_test.index
c = de_scale_data(y_pred_df.iloc[:200,:])

df_plot = pd.concat([a,b,c], axis=1).reset_index(drop=True)
df = df_plot

# Set up the plot
fig, ax = plt.subplots(figsize=(10, 5))
line, = ax.plot([], [], lw=2)
highlight, = ax.plot([], [], color='red', lw=2)  # To highlight the last 5 seconds
highlight_green, = ax.plot([], [], color='green') 
ax.set_xlim(0, df.shape[1] - 1)  # Set x-limits based on the number of columns
ax.set_ylim(df.min().min(), df.max().max())  # Set y-limits based on the data range
ax.set_title('65-Second Time Series Animation')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Value')
ax.grid(True)

# The animate function
def animate(i):
    x = range(65)  # First 60 columns
    y = df.iloc[i].values  # Get the data for the i-th row, only the first 60 columns

    # Update the main line
    line.set_data(x[:-5], y[:-6])

    # Highlight the last 5 seconds (last 5 columns) differently
    highlight.set_data(x[-5:], y[-6:-1])  # Plot the last 5 points in red
    
    new = [y[-1]]*5
    # Highlight the last 10 to 5 seconds in green
    highlight_green.set_data(x[-5:],new)
    
    ax.set_title(f'65-Second Window {i + 1}')
    return line, highlight

# Create the animation
ani = FuncAnimation(fig, animate, frames=len(df), blit=True, interval=1000)

# Close the plot to avoid displaying the static figure
plt.close(fig)

# Display the animation in the notebook
HTML(ani.to_jshtml())

In [73]:
# Now evaluate the performance
y_pred_df

pred_lasso = pd.concat([y_pred_df] * 5, axis=1)

# Rename the columns to range from 0 to 4
pred_lasso.columns = range(5)

# de_scale_data(pred_lasso) 
pred_lasso

,0,1,2,3,4
6252,0.119739,0.119739,0.119739,0.119739,0.119739
4684,0.119739,0.119739,0.119739,0.119739,0.119739
1731,0.119739,0.119739,0.119739,0.119739,0.119739
4742,0.119739,0.119739,0.119739,0.119739,0.119739
4521,0.119739,0.119739,0.119739,0.119739,0.119739
...,...,...,...,...,...
6412,0.119739,0.119739,0.119739,0.119739,0.119739
8285,0.119739,0.119739,0.119739,0.119739,0.119739
7853,0.119739,0.119739,0.119739,0.119739,0.119739
1095,0.119739,0.119739,0.119739,0.119739,0.119739


In [59]:
evaluate_model(y_test_lasso,pred_lasso)

global test nRMSE: 52984.76002826


In [ ]:
x_valid = pd.read_csv("x_valid_process_0.csv")
x_test = pd.read_csv("x_test_process_0.csv")
x_test = x_test.iloc[:,1:]
x_valid = x_valid.iloc[:,1:]
y_valid = pd.read_csv("y_valid.csv",header=None)

In [ ]:
predictions = []
model_filenames = [f"xgb_model_{i}.joblib" for i in range(5)]

for col in range(5):
    model = joblib.load(model_filenames[col])
    
    # Apply the prediction 
    y_pred = model.predict(x_valid)
    predictions.append(y_pred)

# Concatenate predictions for each of the columns into a DataFrame
y_pred_df = pd.DataFrame(predictions).T  # Transpose to align each prediction row-wise
y_pred_df.columns = y.columns  # Name the columns the same as y for easier reference
y_pred_df.index = x_valid.index  # Align the index with X_test for comparison if needed

# Calculate and display the mean squared error for each column
for i, col in enumerate(y.columns):
    mse = mean_squared_error(scale_data(y_valid)[col], y_pred_df[col])
    print(f"Mean Squared Error for {col}: {mse:.2f}")

Mean Squared Error for 0: 0.03
Mean Squared Error for 1: 0.03
Mean Squared Error for 2: 0.03
Mean Squared Error for 3: 0.02
Mean Squared Error for 4: 0.03


In [ ]:
predictions = []
model_filenames = [f"xgb_model_{i}.joblib" for i in range(5)]

for col in range(5):
    model = joblib.load(model_filenames[col])
    
    # Apply the prediction 
    y_pred = model.predict(x_test)
    predictions.append(y_pred)

# Concatenate predictions for each of the columns into a DataFrame
y_pred_df = pd.DataFrame(predictions).T  # Transpose to align each prediction row-wise
y_pred_df.columns = y.columns  # Name the columns the same as y for easier reference
y_pred_df.index = x_test.index  # Align the index with X_test for comparison if needed

In [ ]:
y_pred_df.to_csv("xgboost_on_test.csv")
y_pred_df = pd.read_csv("xgboost_on_test.csv")
y_pred_df = y_pred_df.iloc[:,1:]
y_pred_df = de_scale_data(y_pred_df)
y_pred_df.to_csv("Dauphine_Final_result.csv")